# vehicle speed-detection function using OpenCV Cascade Classifier

In [16]:
import cv2
import numpy as np
import time
import os
os.chdir(r"C:\Users\DT1\Desktop\proj-python\Videos")

In [39]:

cascade_src = r'C:\Users\DT1\Desktop\proj-python\models\cars.xml'

video_src = 'cars_motion2.mp4'

In [45]:

ax1, ay, ax2 = 100, 150, 600
bx1, by, bx2 = 50, 225, 600

def Speed_Cal(t):
    if t <= 0:
        return 0
    return round((10 / t) * 3.6, 2)

i = 1
vehicles = {}
frame_num = 0

cap = cv2.VideoCapture(video_src)
car_cascade = cv2.CascadeClassifier(cascade_src)

fps = cap.get(cv2.CAP_PROP_FPS)
delay = int((1000 / fps) * 1.5)

while True:

    ret, img = cap.read()
    if not ret: break

    frame_num += 1

    img = cv2.resize(img, (640, 480))
    gray = cv2.cvtColor(cv2.blur(img, (15, 15)), cv2.COLOR_BGR2GRAY)
    cars = car_cascade.detectMultiScale(gray, 1.1, 2)

    cv2.line(img, (ax1, ay), (ax2, ay), (255, 0, 0), 2)
    cv2.line(img, (bx1, by), (bx2, by), (255, 0, 0), 2)

    for (x, y, w, h) in cars:

        cx, cy = x + w // 2, y + h // 2

        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255), 2)
        cv2.circle(img, (cx, cy), 5, (0, 255, 0), -1)

        vehicle_id = None

        for vid, data in vehicles.items():

            distance = ((cx - data["cx"]) ** 2 + (cy - data["cy"]) ** 2) ** 0.5

            if distance < 50:
                vehicle_id = vid
                break

        if vehicle_id is None:

            vehicle_id = len(vehicles) + 1

            vehicles[vehicle_id] = {
                "cx": cx,
                "cy": cy,
                "start_frame": None,
                "speed": None
            }

        data = vehicles[vehicle_id]

        old_cy = data["cy"]

        data["cx"] = cx
        data["cy"] = cy

        # Line A

        if data["start_frame"] is None and old_cy < ay <= cy:

            data["start_frame"] = frame_num

            cv2.line(img, (ax1, ay), (ax2, ay), (0, 255, 0), 2)

        # Line B

        if data["start_frame"] is not None and data["speed"] is None and old_cy < by <= cy:

            elapsed_time = (frame_num - data["start_frame"]) / fps

            data["speed"] = Speed_Cal(elapsed_time)

            print(f"Car Number {i} Speed: {data['speed']} KM/H")

            i += 1

            cv2.line(img, (bx1, by), (bx2, by), (0, 255, 0), 2)

        if data["speed"] is not None:

            cv2.putText(img, f"{data['speed']} KM/H", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.putText(img, f"Cars: {i - 1}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow('video', img)

    if cv2.waitKey(delay) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()



Car Number 1 Speed: 89.91 KM/H
Car Number 2 Speed: 77.07 KM/H
Car Number 3 Speed: 89.91 KM/H
Car Number 4 Speed: 89.91 KM/H
Car Number 5 Speed: 6.31 KM/H
